In [1]:
import app
import os
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import box
import rasterio
import rasterio.plot as rplt
import rasterio.windows  as rw
from rasterio.transform import Affine
from matplotlib_scalebar.scalebar import ScaleBar
import numpy as np
import cmocean as cmo

app.setup_logger(use_console_handler=False, use_file_handler=True)

import logging
logger = logging.getLogger(__name__)
logger.setLevel("INFO")

In [2]:
base_directory = r"D:\PhD\21_Experiments\TidesDamageDriver"
drainages_folder = os.path.join(
    base_directory, "02_processed", "02_drainages"
)
data_img_folder = os.path.join(
    base_directory, "01_raw", "L8S2S1-events"
)
data_plot_folder = os.path.join(
    base_directory, "02_processed", "03_drainages_plots"
)

In [3]:
gdf_0 = app.lakes.plotting.get_geodataframes(drainages_folder, ["_e."])[0]
gdf_1 = app.lakes.plotting.get_geodataframes(drainages_folder, ["_1."])[0]

In [4]:
gdf_0.columns

Index(['criteria', 'window', 'lake id', 'type', 'tile', 'ifile_0', 'area',
       'file_0', 'date-0', 'sat-0', 'start-0', 'end-0', 'ifile_1', 'file_1',
       'date-1', 'sat-1', 'start-1', 'end-1', 'lon', 'lat', 'days',
       'days_long', 'area-0', 'status', 'reason', 'median-0', 'std-0',
       'fraction-0', 'fraction_d', 'mean-0', 'volume-0', 'median-1', 'mean-1',
       'volume-1', 'std_depth', 'year', 'geometry'],
      dtype='object')

In [5]:
for row in gdf_0.iterrows():
    if row[0] < 11:
        continue
    if row[0] == 12:
        break
    
    
    centroid = row[1].geometry.centroid
    event_folder = os.path.join(data_img_folder, str(row[0]))
    # count_dpg = len([f for f in os.listdir(event_folder) if "T47DPG.tif" in f])
    # count_dng = len([f for f in os.listdir(event_folder) if "T47DNG.tif" in f])

    # if count_dpg > count_dpg:
    #     filenames = [f for f in os.listdir(event_folder) if "T47DNG.tif" not in f]
    # else:
    #     filenames = [f for f in os.listdir(event_folder) if "T47DPG.tif" not in f]
    filenames = [f for f in os.listdir(event_folder)]
    filenames.sort(key=lambda x: app.lakes.tiffiles.parse_filename(x)[1])  # Sort by date

    ref_date = row[1]["date-0"]
    nerby_optical, nerby_s1 = app.lakes.tiffiles.get_nearby_images(filenames, ref_date, (11, 14), (4, 6))
    logger.info(f"Processing event {row[0]}: {row[1]['file_0']}")
    lake_id = "-".join([str(row[1]['ifile_0']), str(row[1]['ifile_1']), str(int(row[1]['lake id']))])
    try:
        fig, axes = plt.subplots(5, 5, figsize=(22, 15), sharex=True, sharey=True)
        for ifname, fname in enumerate(nerby_optical):
            logger.info(f"Processing optical image {fname}")
            sat, dt = app.lakes.tiffiles.parse_filename(fname)
            ax = axes.ravel()[ifname]
            gpd.GeoDataFrame(geometry=[centroid]).plot(ax=ax, color='red', markersize=5)
            with rasterio.open(os.path.join(event_folder, fname)) as src:

                if sat == 'L8':
                    raster = src.read()
                    transform = src.transform
                elif sat == 'S2':
                    raster, meta = app.lakes.tiffiles.reproject_in_memory(src, "EPSG:3031")
                    transform = meta['transform']
                raster = raster[:3][::-1, :, :]

                rplt.show(raster, ax=ax, transform=transform)
                ax.text(0, 0.95, sat + ' | ' + dt.strftime("%Y-%m-%d %H:%M:%S"), transform=ax.transAxes,
                        fontsize=12, bbox=dict(facecolor='white'))
        
        for ax in axes.ravel():
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_xlabel("")
            ax.set_ylabel("")
            ax.set_title("")
        
        ax = axes.ravel()[0]
        ax.text(0, 0.85,f"{row[0]:02d}" + " | " + lake_id, transform=ax.transAxes, bbox=dict(facecolor='salmon'),
                        fontsize=12)
        
        ax = axes.ravel()[10]
        ax.text(0, 0.85, ref_date, transform=ax.transAxes, bbox=dict(facecolor='salmon'),
                        fontsize=12)
        plt.tight_layout()
        try:
            fig.savefig(os.path.join(data_plot_folder, f"{row[0]:02d}_{lake_id}_optical.png"), dpi=300)
        except Exception as e:
            fig.savefig(os.path.join(data_plot_folder, f"{row[0]:02d}_{str(int(row[1]['lake id']))}_optical.png"), dpi=300)
        plt.close()
    except Exception as e:
        print(f"Error processing event {row[0]}: {e}")
    
    try:
        fig, axes = plt.subplots(2, 5, figsize=(20, 8), sharex=True, sharey=True)
        for ifname, fname in enumerate(nerby_s1):
            sat, dt = app.lakes.tiffiles.parse_filename(fname)
            ax = axes.ravel()[ifname]
            gpd.GeoDataFrame(geometry=[centroid]).plot(ax=ax, color='red', markersize=5)
            with rasterio.open(os.path.join(event_folder, fname)) as src:
                # print(f"Processing {fname}...")
                s1_raster, s1_meta = app.lakes.tiffiles.reproject_in_memory(src, "EPSG:3031")
                s1_transform = s1_meta['transform']
                # raster = src.read()
                # print(src.transform)
                rplt.show(s1_raster, ax=ax, transform=s1_transform, cmap='cmo.ice')
                ax.text(0, 0.95, sat + ' | ' + dt.strftime("%Y-%m-%d %H:%M:%S"), transform=ax.transAxes,
                        fontsize=12, bbox=dict(facecolor='white'))
            # Remove axis labels and ticks
        
        for ax in axes.ravel():
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_xlabel("")
            ax.set_ylabel("")
            ax.set_title("")

        ax = axes.ravel()[0]
        ax.text(0, 0.88,f"{row[0]:02d}" + " | " + lake_id, transform=ax.transAxes, bbox=dict(facecolor='salmon'),
                        fontsize=12)
        ax = axes.ravel()[4]
        ax.text(0, 0.88, ref_date, transform=ax.transAxes, bbox=dict(facecolor='salmon'),
                        fontsize=12)
        plt.tight_layout()
        try:
            fig.savefig(os.path.join(data_plot_folder, f"{row[0]:02d}_{lake_id}_S1.png"), dpi=300)
        except Exception as e:
            fig.savefig(os.path.join(data_plot_folder, f"{row[0]:02d}_{str(int(row[1]['lake id']))}_S1.png"), dpi=300)
        plt.close()
    except Exception as e:
        print(f"Error processing event {row[0]}: {e}")
    